# Cross-Metric Synthesis

This notebook brings together the completed benchmark analyses to address RQ2 and RQ3.

## RQ2
RQ2 examines whether the HTTP performance differences observed under the simple workload persisted, narrowed, widened or changed when the same runtimes were evaluated under the composite workload.

The comparison therefore focuses on:
- HTTP throughput;
- mean HTTP latency;
- P99 HTTP latency;
- runtime differences at 10, 50 and 100 concurrent connections.

No new omnibus significance test is performed across workloads. Instead, the already completed condition-specific statistical results are synthesised using observed performance differences, effect sizes, confidence intervals and adjusted p-values.

## RQ3
RQ3 examines how Node.js, Bun and Deno compare in cold-start time, file I/O performance and memory consumption, and whether these secondary performance dimensions support or contrast with the patterns observed in HTTP performance.

The secondary metrics are therefore compared with the HTTP findings without constructing an aggregate overall-runtime ranking.

In [14]:
library(tidyverse)

current_dir <- normalizePath(
  getwd(),
  winslash = "/"
)

project_root <- if (
  basename(current_dir) == "notebooks"
) {
  dirname(current_dir)
} else {
  current_dir
}

results_root <- file.path(
  project_root,
  "results",
  "tables"
)

project_root

[1] "D:/FOLO_PROJECTS/MASTERS/javascript-runtime-performance-study"

In [15]:
http_throughput_summary <- read_csv(
  file.path(
    results_root,
    "http",
    "http_throughput_descriptive_summary.csv"
  ),
  show_col_types = FALSE
)

http_latency_summary <- read_csv(
  file.path(
    results_root,
    "http",
    "http_latency_descriptive_summary.csv"
  ),
  show_col_types = FALSE
)

http_throughput_inference <- read_csv(
  file.path(
    results_root,
    "http",
    "http_throughput_inferential_results.csv"
  ),
  show_col_types = FALSE
)

http_mean_latency_inference <- read_csv(
  file.path(
    results_root,
    "http",
    "http_mean_latency_inferential_results.csv"
  ),
  show_col_types = FALSE
)

http_p99_latency_inference <- read_csv(
  file.path(
    results_root,
    "http",
    "http_p99_latency_inferential_results.csv"
  ),
  show_col_types = FALSE
)

In [16]:
# inspect loaded data 
names(http_throughput_summary)
names(http_latency_summary)

names(http_throughput_inference)
names(http_mean_latency_inference)
names(http_p99_latency_inference)

[1] "workload"    "connections" "runtime"     "n"           "mean_rps"   
 [6] "median_rps"  "sd_rps"      "se_rps"      "ci_margin"   "ci_lower"   
[11] "ci_upper"    "iqr_rps"     "min_rps"     "max_rps"

[1] "latency_metric" "workload"       "connections"    "runtime"       
 [5] "n"              "mean_ms"        "median_ms"      "sd_ms"         
 [9] "se_ms"          "ci_margin"      "ci_lower"       "ci_upper"      
[13] "iqr_ms"         "min_ms"         "max_ms"

[1] "workload"            "connections"         "group1"             
 [4] "group2"              "test_method"         "estimate"           
 [7] "difference_ci_lower" "difference_ci_upper" "p_holm"             
[10] "significant"         "effect_type"         "effect_size"        
[13] "effect_ci_lower"     "effect_ci_upper"

[1] "workload"            "connections"         "group1"             
 [4] "group2"              "test_method"         "estimate"           
 [7] "difference_ci_lower" "difference_ci_upper" "p_holm"             
[10] "significant"         "effect_type"         "effect_size"        
[13] "effect_ci_lower"     "effect_ci_upper"

[1] "workload"          "connections"       "group1"           
 [4] "group2"            "test_method"       "nonzero_pairs"    
 [7] "median_difference" "median_ci_lower"   "median_ci_upper"  
[10] "p_holm"            "significant"       "effect_size"      
[13] "effect_ci_lower"   "effect_ci_upper"

In [17]:
http_throughput_summary |> head(10)
http_latency_summary |> head(10)

workload,connections,runtime,n,mean_rps,median_rps,sd_rps,se_rps,ci_margin,ci_lower,ci_upper,iqr_rps,min_rps,max_rps
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Simple,10,Node.js,10,16859.00,16823.00,388.0141,122.70085,277.5686,16581.43,17136.57,444.7000,16192.27,17446.00
Simple,10,Bun,10,18049.56,18139.27,605.3813,191.43839,433.0637,17616.50,18482.63,584.6000,16671.87,18841.47
Simple,10,Deno,10,20040.03,19302.81,1594.3582,504.18033,1140.5351,18899.49,21180.56,2635.2300,17969.67,22207.34
Simple,50,Node.js,10,16025.44,15955.74,538.9635,170.43524,385.5513,15639.89,16410.99,477.7625,15065.34,16972.27
Simple,50,Bun,10,19256.60,19350.14,559.8659,177.04513,400.5039,18856.10,19657.10,909.7950,18428.14,20018.27
Simple,50,Deno,10,19018.88,18708.40,1110.9444,351.31146,794.7217,18224.16,19813.61,513.4000,17838.67,21345.47
Simple,100,Node.js,10,15099.96,15398.67,1259.9088,398.41816,901.2845,14198.68,16001.25,581.8350,11643.34,16009.87
Simple,100,Bun,10,17642.22,17766.67,755.7689,238.99510,540.6445,17101.57,18182.86,347.7675,15794.80,18437.20
Simple,100,Deno,10,19009.92,18394.53,1316.2685,416.24065,941.6018,18068.32,19951.52,2396.0700,17740.67,20878.54


latency_metric,workload,connections,runtime,n,mean_ms,median_ms,sd_ms,se_ms,ci_margin,ci_lower,ci_upper,iqr_ms,min_ms,max_ms
<chr>,<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Mean Latency,Simple,10,Node.js,10,0.049,0.050,0.005676462,0.001795055,0.004060696,0.04493930,0.05306070,0.0000,0.04,0.06
Mean Latency,Simple,10,Bun,10,0.119,0.120,0.011972190,0.003785939,0.008564389,0.11043561,0.12756439,0.0100,0.11,0.15
Mean Latency,Simple,10,Deno,10,0.035,0.030,0.007071068,0.002236068,0.005058337,0.02994166,0.04005834,0.0100,0.03,0.05
Mean Latency,Simple,50,Node.js,10,2.746,2.760,0.116351956,0.036793719,0.083233176,2.66276682,2.82923318,0.0650,2.53,2.95
Mean Latency,Simple,50,Bun,10,2.093,2.070,0.078888106,0.024946610,0.056433152,2.03656685,2.14943315,0.1150,1.99,2.23
Mean Latency,Simple,50,Deno,10,2.111,2.130,0.142318110,0.045004938,0.101808243,2.00919176,2.21280824,0.0650,1.83,2.32
Mean Latency,Simple,100,Node.js,10,6.122,5.930,0.718962215,0.227355815,0.514314586,5.60768541,6.63631459,0.2475,5.68,8.13
Mean Latency,Simple,100,Bun,10,5.192,5.140,0.261100236,0.082567144,0.186779857,5.00522014,5.37877986,0.0950,4.94,5.85
Mean Latency,Simple,100,Deno,10,4.850,4.985,0.341402337,0.107960898,0.244224519,4.60577548,5.09422452,0.6100,4.37,5.19


In [18]:
# throughput by runtime

http_throughput_workload_change <- http_throughput_summary |>
  select(
    workload,
    connections,
    runtime,
    mean_rps
  ) |>
  pivot_wider(
    names_from = workload,
    values_from = mean_rps
  ) |>
  mutate(
    absolute_change_rps = Complex - Simple,

    percent_change = (
      (Complex - Simple) / Simple
    ) * 100
  ) |>
  arrange(
    connections,
    runtime
  )

http_throughput_workload_change

connections,runtime,Simple,Complex,absolute_change_rps,percent_change
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
10,Bun,18049.56,15050.24,-2999.327,-16.617171
10,Deno,20040.03,18172.07,-1867.954,-9.321116
10,Node.js,16859.00,12471.48,-4387.527,-26.024831
50,Bun,19256.60,14857.54,-4399.056,-22.844408
50,Deno,19018.88,17952.02,-1066.861,-5.609483
50,Node.js,16025.44,11696.36,-4329.080,-27.013793
100,Bun,17642.22,14989.42,-2652.800,-15.036659
100,Deno,19009.92,17070.44,-1939.485,-10.202487
100,Node.js,15099.96,11546.54,-3553.428,-23.532692


In [19]:
http_throughput_gaps <- http_throughput_summary |>
  select(
    workload,
    connections,
    runtime,
    mean_rps
  ) |>
  pivot_wider(
    names_from = runtime,
    values_from = mean_rps
  ) |>
  transmute(
    workload,
    connections,

    bun_vs_node =
      Bun - `Node.js`,

    deno_vs_node =
      Deno - `Node.js`,

    deno_vs_bun =
      Deno - Bun
  )

http_throughput_gaps

workload,connections,bun_vs_node,deno_vs_node,deno_vs_bun
<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Simple,10,1190.561,3181.022,1990.461
Simple,50,3231.156,2993.441,-237.715
Simple,100,2542.253,3909.959,1367.706
Complex,10,2578.761,5700.595,3121.834
Complex,50,3161.180,6255.660,3094.480
Complex,100,3442.881,5523.902,2081.021


In [20]:
# directly compare simple vs complex 
http_throughput_gap_change <- http_throughput_gaps |>
  pivot_longer(
    cols = c(
      bun_vs_node,
      deno_vs_node,
      deno_vs_bun
    ),
    names_to = "comparison",
    values_to = "throughput_gap_rps"
  ) |>
  pivot_wider(
    names_from = workload,
    values_from = throughput_gap_rps
  ) |>
  mutate(
    gap_change = Complex - Simple
  ) |>
  arrange(
    connections,
    comparison
  )

http_throughput_gap_change

connections,comparison,Simple,Complex,gap_change
<dbl>,<chr>,<dbl>,<dbl>,<dbl>
10,bun_vs_node,1190.561,2578.761,1388.200
10,deno_vs_bun,1990.461,3121.834,1131.373
10,deno_vs_node,3181.022,5700.595,2519.573
50,bun_vs_node,3231.156,3161.180,-69.976
50,deno_vs_bun,-237.715,3094.480,3332.195
50,deno_vs_node,2993.441,6255.660,3262.219
100,bun_vs_node,2542.253,3442.881,900.628
100,deno_vs_bun,1367.706,2081.021,713.315
100,deno_vs_node,3909.959,5523.902,1613.943


In [21]:
# LATENCY WORKLOADS

http_latency_workload_change <- http_latency_summary |>
  filter(
    latency_metric %in%
      c("Mean Latency", "P99 Latency")
  ) |>
  select(
    latency_metric,
    workload,
    connections,
    runtime,
    mean_ms
  ) |>
  pivot_wider(
    names_from = workload,
    values_from = mean_ms
  ) |>
  mutate(
    absolute_change_ms =
      Complex - Simple,

    percent_change =
      ((Complex - Simple) / Simple) * 100
  ) |>
  arrange(
    latency_metric,
    connections,
    runtime
  )

http_latency_workload_change

latency_metric,connections,runtime,Simple,Complex,absolute_change_ms,percent_change
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Mean Latency,10,Bun,0.119,0.165,0.046,38.655462
Mean Latency,10,Deno,0.035,0.069,0.034,97.142857
Mean Latency,10,Node.js,0.049,0.184,0.135,275.510204
Mean Latency,50,Bun,2.093,2.984,0.891,42.570473
Mean Latency,50,Deno,2.111,2.303,0.192,9.095216
Mean Latency,50,Node.js,2.746,3.759,1.013,36.890022
Mean Latency,100,Bun,5.192,6.196,1.004,19.337442
Mean Latency,100,Deno,4.850,5.363,0.513,10.577320
Mean Latency,100,Node.js,6.122,8.188,2.066,33.747141


In [22]:
http_throughput_workload_change
http_throughput_gap_change

http_latency_summary |>
  distinct(latency_metric)

http_latency_workload_change

connections,runtime,Simple,Complex,absolute_change_rps,percent_change
<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
10,Bun,18049.56,15050.24,-2999.327,-16.617171
10,Deno,20040.03,18172.07,-1867.954,-9.321116
10,Node.js,16859.00,12471.48,-4387.527,-26.024831
50,Bun,19256.60,14857.54,-4399.056,-22.844408
50,Deno,19018.88,17952.02,-1066.861,-5.609483
50,Node.js,16025.44,11696.36,-4329.080,-27.013793
100,Bun,17642.22,14989.42,-2652.800,-15.036659
100,Deno,19009.92,17070.44,-1939.485,-10.202487
100,Node.js,15099.96,11546.54,-3553.428,-23.532692


connections,comparison,Simple,Complex,gap_change
<dbl>,<chr>,<dbl>,<dbl>,<dbl>
10,bun_vs_node,1190.561,2578.761,1388.200
10,deno_vs_bun,1990.461,3121.834,1131.373
10,deno_vs_node,3181.022,5700.595,2519.573
50,bun_vs_node,3231.156,3161.180,-69.976
50,deno_vs_bun,-237.715,3094.480,3332.195
50,deno_vs_node,2993.441,6255.660,3262.219
100,bun_vs_node,2542.253,3442.881,900.628
100,deno_vs_bun,1367.706,2081.021,713.315
100,deno_vs_node,3909.959,5523.902,1613.943


latency_metric
<chr>
Mean Latency
P99 Latency


latency_metric,connections,runtime,Simple,Complex,absolute_change_ms,percent_change
<chr>,<dbl>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>
Mean Latency,10,Bun,0.119,0.165,0.046,38.655462
Mean Latency,10,Deno,0.035,0.069,0.034,97.142857
Mean Latency,10,Node.js,0.049,0.184,0.135,275.510204
Mean Latency,50,Bun,2.093,2.984,0.891,42.570473
Mean Latency,50,Deno,2.111,2.303,0.192,9.095216
Mean Latency,50,Node.js,2.746,3.759,1.013,36.890022
Mean Latency,100,Bun,5.192,6.196,1.004,19.337442
Mean Latency,100,Deno,4.850,5.363,0.513,10.577320
Mean Latency,100,Node.js,6.122,8.188,2.066,33.747141


### RQ2 Descriptive Synthesis: Effect of Workload Complexity

Moving from the simple fixed-response HTTP workload to the composite workload reduced throughput for all three runtimes at every concurrency level. However, the magnitude of this reduction differed substantially between runtimes.

Deno showed the smallest throughput degradation at all three connection levels: approximately 9.3% at 10 connections, 5.6% at 50 connections and 10.2% at 100 connections. Bun declined by approximately 16.6%, 22.8% and 15.0%, respectively, while Node.js showed the largest reductions at 26.0%, 27.0% and 23.5%.

The between-runtime throughput gaps therefore generally persisted or widened rather than disappearing as workload complexity increased. Deno's advantage over Node.js increased from approximately 3,181 to 5,701 requests/s at 10 connections, from 2,993 to 6,256 requests/s at 50 connections, and from 3,910 to 5,524 requests/s at 100 connections. At 50 connections, the descriptive Bun-Deno ordering also changed: Bun was approximately 238 requests/s ahead under the simple workload, whereas Deno was approximately 3,094 requests/s ahead under the composite workload.

Mean latency increased under the composite workload for all three runtimes. In absolute terms, Deno showed the smallest increase at every concurrency level: 0.034 ms at 10 connections, 0.192 ms at 50 connections and 0.513 ms at 100 connections. Node.js showed the largest increase at each level. Percentage changes at very low baseline latencies were interpreted cautiously because small absolute changes can produce very large percentages.

The mean-latency ordering also changed under some conditions. At 10 connections, Node.js had lower mean latency than Bun under the simple workload, but Bun had lower latency than Node.js under the composite workload. At 50 connections, Bun and Deno were descriptively very close under the simple workload, whereas Deno developed a clear latency advantage under the composite workload. At 100 connections, the ordering remained Deno, Bun and Node.js from lowest to highest mean latency, but the separation increased.

P99 latency showed a less uniform pattern. Deno's P99 latency remained unchanged at 10 connections and increased less than the other runtimes at 50 connections. At 100 connections, however, Deno's P99 latency increased by 4.9 ms, compared with 2.4 ms for Bun and 3.7 ms for Node.js. Consequently, Bun recorded lower P99 latency than Deno under the complex 100-connection condition despite Deno retaining lower mean latency.

Overall, workload complexity did not cause the HTTP performance differences to converge consistently. Instead, the relative differences often persisted or widened, while the precise pattern depended on the performance measure and concurrency level.

In [23]:
rq2_throughput_evidence <- http_throughput_inference |>
  select(
    workload,
    connections,
    group1,
    group2,
    estimate,
    difference_ci_lower,
    difference_ci_upper,
    p_holm,
    significant,
    effect_type,
    effect_size
  ) |>
  arrange(
    connections,
    group1,
    group2,
    workload
  )

rq2_mean_latency_evidence <- http_mean_latency_inference |>
  select(
    workload,
    connections,
    group1,
    group2,
    estimate,
    difference_ci_lower,
    difference_ci_upper,
    p_holm,
    significant,
    effect_type,
    effect_size
  ) |>
  arrange(
    connections,
    group1,
    group2,
    workload
  )

rq2_p99_evidence <- http_p99_latency_inference |>
  select(
    workload,
    connections,
    group1,
    group2,
    median_difference,
    median_ci_lower,
    median_ci_upper,
    p_holm,
    significant,
    effect_size
  ) |>
  arrange(
    connections,
    group1,
    group2,
    workload
  )

In [24]:
rq2_throughput_evidence
rq2_mean_latency_evidence
rq2_p99_evidence

workload,connections,group1,group2,estimate,difference_ci_lower,difference_ci_upper,p_holm,significant,effect_type,effect_size
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
Complex,10,Bun,Deno,-3121.834,-3461.562,-2782.1061,9.039788e-08,Yes,Cohen's dz,-6.5735703
Simple,10,Bun,Deno,-1990.461,-3016.337,-964.5849,1.747682e-02,Yes,Cohen's dz,-1.3879747
Complex,10,Node.js,Bun,-2578.761,-2843.323,-2314.1993,5.750537e-08,Yes,Cohen's dz,-6.9727959
Simple,10,Node.js,Bun,-1190.561,-1561.099,-820.0227,5.666073e-04,Yes,Cohen's dz,-2.2984834
Complex,10,Node.js,Deno,-5700.595,-5998.751,-5402.4395,1.602476e-10,Yes,Cohen's dz,-13.6772912
Simple,10,Node.js,Deno,-3181.022,-4291.244,-2070.8003,1.252555e-03,Yes,Cohen's dz,-2.0496501
Complex,50,Bun,Deno,-3094.480,-3450.183,-2738.7767,1.360789e-07,Yes,Cohen's dz,-6.2233265
Simple,50,Bun,Deno,670.940,-699.200,987.7350,4.921875e-01,No,Rank-biserial correlation,0.2727273
Complex,50,Node.js,Bun,-3161.180,-3430.080,-2892.2801,1.162759e-08,Yes,Cohen's dz,-8.4097175


workload,connections,group1,group2,estimate,difference_ci_lower,difference_ci_upper,p_holm,significant,effect_type,effect_size
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<chr>,<dbl>
Complex,10,Bun,Deno,0.096,8.831014e-02,0.10368986,7.236007e-09,Yes,Cohen's dz,8.9305009
Simple,10,Bun,Deno,0.080,8.000000e-02,0.09500000,2.539062e-02,Yes,Rank-biserial correlation,1.0000000
Complex,10,Node.js,Bun,0.019,7.101254e-03,0.03089875,2.539062e-02,Yes,Cohen's dz,1.1422869
Simple,10,Node.js,Bun,-0.070,-7.500000e-02,-0.06500000,2.539062e-02,Yes,Rank-biserial correlation,-1.0000000
Complex,10,Node.js,Deno,0.115,1.005938e-01,0.12940615,3.122955e-07,Yes,Cohen's dz,5.7104806
Simple,10,Node.js,Deno,0.015,1.000000e-02,0.02000000,2.539062e-02,Yes,Rank-biserial correlation,1.0000000
Complex,50,Bun,Deno,0.681,6.183229e-01,0.74367713,2.342438e-08,Yes,Cohen's dz,7.7725007
Simple,50,Bun,Deno,-0.060,-1.200000e-01,0.10500000,5.390625e-01,No,Rank-biserial correlation,-0.2363636
Complex,50,Node.js,Bun,0.775,6.932105e-01,0.85678949,7.384989e-08,Yes,Cohen's dz,6.7783968


workload,connections,group1,group2,median_difference,median_ci_lower,median_ci_upper,p_holm,significant,effect_size
<chr>,<dbl>,<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<chr>,<dbl>
Complex,10,Bun,Deno,2.0,2.0,3.0,0.03515625,Yes,1.0000000
Simple,10,Bun,Deno,2.0,2.0,2.0,0.03515625,Yes,1.0000000
Complex,10,Node.js,Bun,-2.0,-2.5,-2.0,0.03515625,Yes,-1.0000000
Simple,10,Node.js,Bun,-2.0,-2.0,-2.0,0.03515625,Yes,-1.0000000
Complex,10,Node.js,Deno,0.0,0.0,0.0,1.00000000,No,1.0000000
Simple,10,Node.js,Deno,0.0,0.0,0.0,1.00000000,No,NA
Complex,50,Bun,Deno,1.0,0.5,2.0,0.07031250,No,1.0000000
Simple,50,Bun,Deno,0.0,0.0,1.0,0.62500000,No,1.0000000
Complex,50,Node.js,Bun,1.0,0.0,1.0,0.21875000,No,1.0000000


### RQ2 Synthesis: Do Runtime Differences Persist as Workload Complexity Increases?

The descriptive and inferential results indicate that increasing HTTP workload complexity did not generally eliminate the performance differences between Node.js, Bun and Deno. Instead, the effect of the composite workload depended on the runtime, concurrency level and performance measure.

For **HTTP throughput**, all three runtimes processed fewer requests per second under the composite workload, but the reduction was not uniform. Deno showed the smallest throughput decline at all three concurrency levels, while Node.js showed the largest. Consequently, Deno's descriptive throughput advantage over Node.js widened from approximately 3,181 to 5,701 requests/s at 10 connections, from 2,993 to 6,256 requests/s at 50 connections, and from 3,910 to 5,524 requests/s at 100 connections.

The inferential results supported this pattern. Under the simple workload, 17 of the 18 pairwise throughput comparisons were statistically significant after Holm correction; the only non-significant comparison was Bun versus Deno at 50 connections. Under the composite workload, all runtime pairs differed significantly at every connection level. Particularly at 50 connections, the simple workload provided no statistically significant evidence of a Bun-Deno throughput difference, whereas the composite workload produced a clear Deno advantage. This suggests that increased processing complexity exposed a performance distinction that was not evident under the simpler condition.

For **mean latency**, the composite workload increased latency for all runtimes. Deno had the smallest absolute increase at each connection level, whereas Node.js showed the largest. The inferential results again showed stronger separation under the composite workload. Bun and Deno were not significantly different at 50 or 100 connections under the simple workload, but these comparisons became statistically significant under the composite workload. At 10 connections, the relative position of Node.js and Bun also changed: Node.js had lower mean latency under the simple workload, whereas Bun had lower mean latency under the composite workload. Thus, the mean-latency results provide further evidence that additional workload processing did not force the runtimes towards equivalent performance.

The **P99 latency** results were more workload- and concurrency-dependent. At 10 connections, Bun had significantly higher P99 latency than both Deno and Node.js under both workload types, while Node.js and Deno were not significantly different. At 50 connections, most P99 pairwise differences remained statistically non-significant; only the Node.js-Deno comparison reached significance under the composite workload. Therefore, the descriptive differences at this level should not be interpreted as strong evidence of complete runtime separation.

The clearest tail-latency change occurred at **100 connections**. Under the simple workload, none of the three P99 pairwise comparisons was statistically significant after Holm correction. Under the composite workload, all three comparisons became significant. Bun recorded the lowest P99 latency, followed by Deno and then Node.js. This differs from the mean-latency result, where Deno retained the lowest average latency. The distinction demonstrates that a runtime can perform strongly on average request latency while displaying a different pattern in its slowest requests under high concurrency.

Overall, RQ2 is answered by concluding that the HTTP performance disparities **generally persisted and, for throughput and mean latency, often became more pronounced under the composite workload rather than converging**. However, the exact behaviour was metric-dependent. In particular, P99 latency showed that increased complexity could alter the relative runtime pattern at high concurrency rather than simply magnifying the simple-workload result.

In [25]:
# RQ 3 SRTS

cold_start_summary <- read_csv(
  file.path(
    results_root,
    "cold_start",
    "cold_start_descriptive_summary.csv"
  ),
  show_col_types = FALSE
)

cold_start_inference <- read_csv(
  file.path(
    results_root,
    "cold_start",
    "cold_start_inferential_results.csv"
  ),
  show_col_types = FALSE
)

file_io_duration_summary <- read_csv(
  file.path(
    results_root,
    "file_io",
    "file_io_duration_descriptive_summary.csv"
  ),
  show_col_types = FALSE
)

file_io_inference <- read_csv(
  file.path(
    results_root,
    "file_io",
    "file_io_inferential_results.csv"
  ),
  show_col_types = FALSE
)

memory_summary <- read_csv(
  file.path(
    results_root,
    "memory",
    "memory_median_rss_descriptive_summary.csv"
  ),
  show_col_types = FALSE
)

memory_inference <- read_csv(
  file.path(
    results_root,
    "memory",
    "memory_inferential_results.csv"
  ),
  show_col_types = FALSE
)

In [26]:
cold_start_summary
file_io_duration_summary
memory_summary

runtime,n,mean_ms,median_ms,sd_ms,se_ms,ci_margin,ci_lower,ci_upper,iqr_ms,min_ms,max_ms
<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Node.js,30,160.7457,154.3487,17.72058,3.235320,6.616972,154.1288,167.3627,15.461325,141.4879,208.3294
Bun,30,196.0424,187.7892,25.82093,4.714235,9.641693,186.4007,205.6841,16.190700,169.1882,283.0018
Deno,30,140.2175,132.0528,36.07831,6.586969,13.471864,126.7456,153.6894,8.323725,115.7602,281.6167


operation,runtime,n,mean_ms,median_ms,sd_ms,se_ms,ci_margin,ci_lower,ci_upper,iqr_ms,min_ms,max_ms
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Read,Node.js,10,91.58301,94.2139,4.559993,1.441996,3.262022,88.32099,94.84503,6.264825,82.1170,95.7881
Read,Bun,10,105.19027,104.5327,3.565541,1.127523,2.550635,102.63964,107.74090,5.333775,100.9047,111.8070
Read,Deno,10,137.44535,136.7507,3.048423,0.963996,2.180710,135.26464,139.62606,2.392950,133.6609,144.4641
Write,Node.js,10,63.17455,61.2453,4.794998,1.516312,3.430135,59.74442,66.60468,3.727250,58.1735,74.3964
Write,Bun,10,1069.45525,1064.5662,14.655075,4.634342,10.483609,1058.97164,1079.93886,9.048550,1056.4500,1101.1186
Write,Deno,10,1068.63017,1058.3490,35.898562,11.352122,25.680284,1042.94989,1094.31045,19.636575,1040.5130,1163.9381


workload,runtime,n,mean_mib,median_mib,sd_mib,se_mib,ci_margin,ci_lower,ci_upper,iqr_mib,min_mib,max_mib
<chr>,<chr>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
Idle,Node.js,20,36.70469,36.69531,0.04277956,0.009565801,0.020021452,36.68467,36.72471,0.05078125,36.61328,36.78125
Idle,Bun,20,55.91328,55.91406,0.01486646,0.003324241,0.006957716,55.90632,55.92024,0.02441406,55.88672,55.93750
Idle,Deno,20,42.39531,42.42773,0.11799747,0.026385038,0.055224518,42.34009,42.45054,0.10839844,42.13281,42.57812
Allocated,Node.js,20,137.29707,137.28906,0.02795300,0.006250482,0.013082409,137.28399,137.31015,0.03710938,137.25781,137.36719
Allocated,Bun,20,155.99160,156.00195,0.05410099,0.012097349,0.025320041,155.96628,156.01692,0.01562500,155.76953,156.04688
Allocated,Deno,20,142.90625,142.93945,0.11904369,0.026618978,0.055714161,142.85054,142.96196,0.10156250,142.63281,143.09375


### RQ3 Cross-Metric Summary

| Performance dimension | Main observed pattern | Relationship to HTTP findings |
|---|---|---|
| Cold start | Deno fastest, followed by Node.js, then Bun | Broadly supports Deno's strong HTTP performance |
| File read | Node.js fastest, followed by Bun, then Deno | Contrasts with HTTP, where Node.js was often slower |
| File write | Node.js substantially faster; Bun and Deno not significantly different | Strongly contrasts with the HTTP pattern |
| Memory – Idle | Node.js lowest RSS, followed by Deno, then Bun | Contrasts with HTTP by favouring Node.js |
| Memory – Allocated | Node.js lowest RSS, followed by Deno, then Bun | Same contrast as Idle memory |

### RQ3 Synthesis: Secondary Performance Metrics

The secondary performance metrics did not reproduce a single consistent version of the HTTP performance pattern. Instead, cold-start time, file I/O and memory usage revealed different strengths among the three runtimes.

For **cold-start performance**, Deno recorded the lowest mean start-up time at approximately 140.22 ms, followed by Node.js at 160.75 ms and Bun at 196.04 ms. The Friedman test and subsequent pairwise comparisons showed statistically significant differences between all three runtimes. This result broadly supports the HTTP findings in which Deno frequently demonstrated strong throughput and mean-latency performance. However, Bun's comparatively strong HTTP performance did not extend to cold start, where it was the slowest of the three runtimes.

The **file I/O results contrasted more clearly with HTTP performance**. For file reading, Node.js recorded the lowest mean duration at approximately 91.58 ms, compared with 105.19 ms for Bun and 137.45 ms for Deno. All three pairwise differences were statistically significant. Thus, Deno's strong HTTP and cold-start performance did not extend to the controlled whole-file read workload.

The contrast was greater for **file writing**. Node.js completed the 100 MiB write operation in approximately 63.17 ms on average, whereas Bun and Deno required approximately 1069.46 ms and 1068.63 ms, respectively. Node.js differed significantly from both runtimes, while the Bun-Deno difference was not statistically significant. This represents a substantially different performance pattern from the HTTP benchmarks and demonstrates that strong HTTP performance cannot be assumed to imply strong file-system performance.

The **memory results also favoured Node.js rather than the runtime that most frequently led the HTTP throughput results**. Under the Idle workload, mean median post-READY RSS was approximately 36.70 MiB for Node.js, 42.40 MiB for Deno and 55.91 MiB for Bun. Under the Allocated workload, the corresponding means were approximately 137.30 MiB, 142.91 MiB and 155.99 MiB. All runtime pairs differed significantly under both workloads, and the ordering Node.js, Deno and Bun from lowest to highest memory consumption was consistent across all matched repetitions.

Taken together, the secondary metrics therefore **both support and contradict aspects of the HTTP findings rather than confirming a single universal runtime ordering**. Deno's strong HTTP performance was accompanied by the fastest cold-start performance, providing one area of cross-metric consistency. In contrast, Node.js performed substantially better in file I/O and consumed the least memory despite often showing lower HTTP throughput and higher mean latency than Bun or Deno.

RQ3 therefore indicates that the relative performance of Node.js, Bun and Deno is strongly dependent on the performance dimension being considered. HTTP results alone would not provide a complete basis for runtime selection: applications that prioritise request processing, rapid process start-up, file-intensive operations or memory efficiency may favour different runtime characteristics.

In [27]:
synthesis_results_directory <- file.path(
  project_root,
  "results",
  "tables",
  "synthesis"
)

dir.create(
  synthesis_results_directory,
  recursive = TRUE,
  showWarnings = FALSE
)

write_csv(
  http_throughput_workload_change,
  file.path(
    synthesis_results_directory,
    "rq2_throughput_workload_change.csv"
  )
)

write_csv(
  http_throughput_gap_change,
  file.path(
    synthesis_results_directory,
    "rq2_throughput_gap_change.csv"
  )
)

write_csv(
  http_latency_workload_change,
  file.path(
    synthesis_results_directory,
    "rq2_latency_workload_change.csv"
  )
)

In [28]:
list.files(synthesis_results_directory)

[1] "rq2_latency_workload_change.csv"    "rq2_throughput_gap_change.csv"     
[3] "rq2_throughput_workload_change.csv"